## ___Multi Response Phylogenetic Mixed Models `MR-PMM` using Markov Chain Monte Carlo generalized linear mixed models `MCMCglmm`___
----------------------

In [2]:
# multi response phylogenetic mixed effect models - look up https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_euc_example_analysis.html

In [1]:
set.seed(2026 - 3 - 9)

suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("MCMCglmm")
    library("brms")
    library("nlme")
    library("phylolm")
})

In [2]:
data <- read.csv("../../data/chapter2/FREDv3subset/collab_fineroots_log_995_species_means_5states_name_matched_with_phylogeny.csv") # log transformed SRL and RD species averages
phylogeny <- ape::read.tree("../../data/chapter2/uphylomaker/collab_fineroots_log_995_species_means_5states.tre") # phylogenetic tree
stopifnot(data$binominal==phylogeny$tip.label) # make sure the binominal names are matched between the trait data and the phylogeny

In [3]:
ape::is.binary(phylogeny) # damn

[1] FALSE

In [4]:
ape::is.ultrametric(phylogeny) # :)

[1] TRUE

In [5]:
phylogeny <- ape::multi2di(phylogeny) # make the phylogeny completely bifurcating by introducing 0 length branches
sum(phylogeny$edge.length == 0) # damn

[1] 91

In [6]:
phylogeny$edge.length[phylogeny$edge.length==0] <- rnorm(n = sum(phylogeny$edge.length == 0), mean = 1e-6, sd = 1e-8) # replace the 0 length edges with random noise
sum(phylogeny$edge.length == 0) # no more 0 length branches

[1] 0

In [7]:
# "By including more diverse species in our phylogeny, we capture deeper splits that represent more meaningful divergences in the genotype and phenotype of extant lineages,
# precisely the effects we intend to model when analysing inter-species data."
# "One consequence of this, is that shallow topology (near the tips) is less informative than deep topology when attempting to infer patterns of phylogenetic niche conservatism, because differences between genera
# are usually more significant than differences between species within genera."

In [8]:
# "For higher taxonomic ranks (e.g. genus), it will usually be possible to derive a unique consensus tree by sampling a single species from each genus and simply pruning off the other tips from the tree.
# This approach may be problematic for lower taxonomic ranks however, because more closely related species are less likely to be monophyletic with respect to the taxonomic rank in question.
# Even in such cases, we can easily account for this phylogenetic uncertainty by randomly sampling topologies at the specified rank and fitting our models over this sample of trees."

In [9]:
# harvest the genus names
gsub(phylogeny$tip.label, pattern = "_[a-z]+", replacement = '')[1:10]

[1] "Rudbeckia"  "Ratibida"   "Heliopsis"  "Liatris"    "Arnica"    
 [6] "Arnica"     "Hymenoxys"  "Helianthus" "Helianthus" "Helianthus"

In [10]:
setdiff(gsub(phylogeny$tip.label, pattern = "_[a-z]+", replacement = ''), unique(data$F01286)) # regex needs changes

[1] "Symphyotrichum-angliae" "Festuca-bernardii"      "Laurelia-zelandiae"    
[4] "Blechnum-zelandiae"

In [11]:
data[data$F01286 == "Laurelia", ] # the hyphen is part of the specific epithet

,binominal,F01286,F01287,F01289,F01290,F00056,F00004,F00679,F00727,state
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>
896,Laurelia_novae-zelandiae,Laurelia,novae-zelandiae,Atherospermataceae,Laurales,NA,"Kramer-Walter KR, Bellingham PJ, Millar TR, Smissen RD, Richardson SJ, Laughlin DC. 2016. Root traits are multidimensional: specific root length is independent from root tissue density and the plant economic spectrum. Journal of Ecology 104: 1299-1310.",-0.04908521,2.371937,AM


In [12]:
# data[data$F01286 == "Leucaena", ] # we have a specific epithet starting with a capital?????

In [14]:
# unique(gsub(phylogeny$tip.label, pattern = "_[-a-z]+", replacement = '', fixed = FALSE))

In [15]:
setdiff(unique(gsub(phylogeny$tip.label, pattern = "_[-a-z]+", replacement = '', fixed = FALSE)), unique(data$F01286)) # good :)

character(0)

In [16]:
# sample the phylogeny repeatedly, with one randomly chosen species per genera

NUNIQUE_GENERA <- length(unique(data$F01286)) # number of unique genera in our phylogeny
NSAMPLES = 100 # number of times to sample the original phylogeny
sampled_phylogenies <- list() # sampled sub phylogeneies with genera at tips
sampled_vcvs <- list()

for (i in 1:NSAMPLES) {
    sampled_species <- mapply(split.data.frame(data[, c("binominal", "F01286")], ~F01286), FUN = function(df) sample(df$binominal, 1)) # this is a vector of one randomly sampled species per each genera in the phylogeny
    subphylogeny <- ape::keep.tip(phy = phylogeny, tip = unname(sampled_species), trim.internal = TRUE) # trimmed phylogeny with one randomly sampled species per genus
    stopifnot(length(subphylogeny$tip.label)==NUNIQUE_GENERA)
                                     
    if(!ape::is.binary(subphylogeny)) subphylogeny <- ape::multi2di(subphylogeny) # make bifucracting if not already
    if(!ape::is.ultrametric(subphylogeny)) subphylogeny <- phytools::force.ultrametric(subphylogeny, method = "extend", message = FALSE) # make ultrametric if not already

    subphylogeny$tip.label <- gsub(subphylogeny$tip.label, pattern = "_[-a-z]+", replacement = '', fixed = FALSE) # replace the tip labels (binominal names) with genus names
    stopifnot(length(setdiff(subphylogeny$tip.label, unique(data$F01286))) == 0)
                              
    sampled_phylogenies[[i]] <- subphylogeny
    sampled_vcvs[[i]] <- ape::vcv.phylo(phy = subphylogeny, corr = TRUE) # computes the expected variances and covariances of a continuous phenotype assuming it evolves under a Brownian motion model
}

stopifnot(length(sampled_phylogenies) == length(sampled_vcvs))

In [ ]:
# create list of C matrices and dfs for use with brm_multiple() - DONT KNOW WTF THESE ARE????


In [17]:
# first try the traditional OLS and PGLS approaches for the selected root traits (RD, SRL)
head(data)

,binominal,F01286,F01287,F01289,F01290,F00056,F00004,F00679,F00727,state
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<chr>
1,Rudbeckia_hirta,Rudbeckia,hirta,Asteraceae,Asterales,NA,"Poon GT, Maherali H. 2015. Competitive interactions between a nonmycorrhizal invasive plant, Alliaria petiolata, and a suite of mycorrhizal grassland, old field, and forest species. PeerJ 3: e1090",-1.3327915,5.694268,AM
2,Ratibida_pinnata,Ratibida,pinnata,Asteraceae,Asterales,NA,"Craine JM, Froehle J, Tilman DG, Wedin DA, Chapin FS. 2001. The relationships among root and leaf traits of 76 grassland species and relative abundance among fertility and disturbance gradients. Oikos 93: 274-285",-0.8209806,4.025352,AM
3,Heliopsis_helianthoides,Heliopsis,helianthoides,Asteraceae,Asterales,NA,"Craine JM, Froehle J, Tilman DG, Wedin DA, Chapin FS. 2001. The relationships among root and leaf traits of 76 grassland species and relative abundance among fertility and disturbance gradients. Oikos 93: 274-285",-0.9675840,4.158883,NM
4,Liatris_aspera,Liatris,aspera,Asteraceae,Asterales,NA,"Craine JM, Froehle J, Tilman DG, Wedin DA, Chapin FS. 2001. The relationships among root and leaf traits of 76 grassland species and relative abundance among fertility and disturbance gradients. Oikos 93: 274-285",-0.7985077,4.382027,AM
5,Arnica_sororia,Arnica,sororia,Asteraceae,Asterales,NA,"Kembel SW, Cahill JF, Jr. 2011. Independent Evolution of Leaf and Root Traits within and among Temperate Grassland Plant Communities. PLoS ONE 6(6): e19992.",-1.4134600,4.342257,AM
6,Arnica_fulgens,Arnica,fulgens,Asteraceae,Asterales,NA,"Kembel SW, Cahill JF, Jr. 2011. Independent Evolution of Leaf and Root Traits within and among Temperate Grassland Plant Communities. PLoS ONE 6(6): e19992.",-0.6627150,2.765172,AM


In [18]:
# the tutorial uses the R library phylolm, but why????? could've just used nlme::gls
# let's try both

In [19]:
# nlme::gls
corrmat <- ape::corBrownian(phy = phylogeny) # transform the phylogeny into a correlation matrix??
pgls_0 <- nlme::gls(F00727 ~ F00679, data = data, correlation = corrmat) # log(SRL) ~ log(RD) (RD as the predictor of SRL)

Warning message in Initialize.corPhyl(X[[i]], ...):
"No covariate specified, species will be taken as ordered in the data frame. To avoid this message, specify a covariate containing the species names with the 'form' argument."


In [20]:
# don't worry about the waring as our data and phylogeny has been name matched
summary(pgls_0)

Generalized least squares fit by REML
  Model: F00727 ~ F00679 
  Data: data 
       AIC      BIC    logLik
  3807.344 3822.046 -1900.672

Correlation Structure: corBrownian
 Formula: ~1 
 Parameter estimate(s):
numeric(0)

Coefficients:
                Value Std.Error    t-value p-value
(Intercept)  2.603773 3.1017055   0.839465  0.4014
F00679      -1.254196 0.0734152 -17.083601  0.0000

 Correlation: 
       (Intr)
F00679 0.022 

Standardized residuals:
         Min           Q1          Med           Q3          Max 
-1.089832941 -0.073093691 -0.008224465  0.065609870  0.329875443 

Residual standard error: 7.361376 
Degrees of freedom: 995 total; 993 residual

In [24]:
# using phylolm

data_phylolm <- data # making a copy
rownames(data_phylolm) <- data_phylolm$binominal

pgls_1_brownian <- phylolm::phylolm(F00727 ~ F00679, data = data_phylolm, phy = phylogeny, model = "BM") # lambda fixed to 1
pgls_1_lambda <- phylolm::phylolm(F00727 ~ F00679, data = data_phylolm, phy = phylogeny, model = "lambda") # lambda estimated - don't understand how this works through!!!

In [25]:
summary(pgls_1_brownian) # this is very similar to what we got from nlme::gls()


Call:
phylolm::phylolm(formula = F00727 ~ F00679, data = data_phylolm, 
    phy = phylogeny, model = "BM")

   AIC logLik 
  3808  -1901 

Raw residuals:
    Min      1Q  Median      3Q     Max 
-8.0227 -0.5381 -0.0605  0.4830  2.4283 

Mean tip height: 400.7877
Parameter estimate(s) using ML:
sigma2: 0.1349366 

Coefficients:
             Estimate    StdErr  t.value p.value    
(Intercept)  2.603773  3.101705   0.8395  0.4014    
F00679      -1.254196  0.073415 -17.0836  <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

R-squared: 0.2271	Adjusted R-squared: 0.2264 

In [26]:
summary(pgls_1_lambda)


Call:
phylolm::phylolm(formula = F00727 ~ F00679, data = data_phylolm, 
    phy = phylogeny, model = "lambda")

   AIC logLik 
  2449  -1221 

Raw residuals:
    Min      1Q  Median      3Q     Max 
-8.2950 -0.4969 -0.0456  0.4195  2.6135 

Mean tip height: 400.7877
Parameter estimate(s) using ML:
lambda : 0.6740991
sigma2: 0.003957272 

Coefficients:
             Estimate    StdErr  t.value   p.value    
(Intercept)  2.269711  0.467750   4.8524 1.416e-06 ***
F00679      -1.569916  0.063468 -24.7355 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

R-squared: 0.3812	Adjusted R-squared: 0.3806 

Note: p-values and R-squared are conditional on lambda=0.6740991.